# Simple Two-Layer Piezo Patch Optimizer — Single Mode, Tip Deflection

This notebook is intentionally stripped down to the minimum pieces needed to start optimization:

1. **Outer layer:** optimize patch geometry using patch lengths and gaps.
2. **Inner layer:** for each candidate geometry, optimize actuation for one target mode by searching frequency and phase/sign pattern.
3. **FE update:** every geometry candidate builds a new region-based Euler--Bernoulli FE model using `build_geometry_with_regions(...)`.

The objective is the best single-mode **tip displacement per volt**:

\[
J(\mathbf g)=\max_{\omega\in\Omega_m,\,\mathbf s\in\{-1,+1\}^{N_p}}
\left|e_{tip}^T\left(K(\mathbf g)+i\omega C(\mathbf g)-\omega^2M(\mathbf g)\right)^{-1}\Gamma(\mathbf g)\mathbf s\right|.
\]

This notebook uses explicit beam regions: `substrate`, `kapton`, and `piezo`.


## 0. Imports

Run this notebook from your project root, i.e. the directory that contains `Modeling/`.


In [ ]:
from __future__ import annotations

import copy
import itertools
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution

# -------------------------------------------------------------------------
# EDIT THIS if imports fail.
# Usually this should be your project root, i.e. the folder containing Modeling/.
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parents[2]
sys.path.append(str(PROJECT_ROOT))

from Modeling.models.beam_properties import PiezoBeamParams, compute_EI_and_rhoA
import Modeling.models.FE3 as FE_module

print("Imported FE_module from:", FE_module.__file__)


Imported FE_module from: c:\Users\dlie9\OneDrive - Georgia Institute of Technology\General - Swimmers\Software\Arduino\Arduino Playground\FishCode\multi_mfc_control\Modeling\models\FE3.py


## 1. Beam/region material properties, Base FE/coupling parameters

This section follows the idea from the arbitrary-placement notebook: compute section properties for each region type and pass those region properties into `build_geometry_with_regions(...)`.

Edit this section to match your actual beam/patch/Kapton dimensions and materials.

The explicit region model controls `EI` and `rhoA`. `PiezoBeamParams` is still used for damping and electromechanical coupling `theta_mech`.

Because geometry is supplied manually, the default patch array inside `PiezoBeamParams` is ignored.

In [ ]:
# ----------------------------
# Editable beam dimensions
# ----------------------------
L = 0.365          # total beam length [m]
b_s = 50e-3      # substrate width [m]
b_p = 14e-3       # active piezo width [m]

h_s = 0.504e-3     # substrate thickness [m]
h_p = 0.3e-3     # one piezo layer thickness [m]

rho_s = 1190.0    # substrate density [kg/m^3]
rho_p = 7800.0    # piezo density [kg/m^3]

E_s = 3.0e9       # substrate Young's modulus [Pa]
E_p = 31.0e9      # piezo Young's modulus [Pa]


base_params = PiezoBeamParams(
    # ===================== Geometry =====================
	w_p = b_p,			# patch width [m]
	w_s = 0.265625e-3,		# spacing between patches [m]
	n_patches = 31,				# number of unit cells (patches)
	b = b_p,			# beam width [m]
	hp = h_p,			# piezo thickness [m]
	hs = h_s,			# substrate thickness [m]

	# ===================== Materials =====================
	rho_p = rho_p,			# piezo density [kg/m^3]
	rho_s = rho_s,			# substrate density [kg/m^3]
	E_s = E_s,			# substrate Young's modulus [Pa]
	s11 = 1.0 / E_p,			# piezo compliance [m^2/N]
	d31 = -1.48e-10,			# piezo strain constant [C/N]
	eps_r = 1700,			# relative permittivity
	omega_p = 2*np.pi*1,
	omega_q = 2*np.pi*100,
)

# Damping calibration values. Edit these freely.
base_params.zeta_p = 0.05
base_params.zeta_q = 0.20

# ----------------------------
# Region section properties
# ----------------------------
EI_substrate = b_s * E_s * h_s**3 / 12.0
rhoA_substrate = b_s * rho_s * h_s

EI_piezo, rhoA_piezo = compute_EI_and_rhoA(
    E_layers=[E_s, E_p],
    rho_layers=[rho_s, rho_p],
    h_layers=[h_s, h_p],
    b_layers=[b_s, b_p],
)

region_types = {
    "substrate": {"EI": EI_substrate, "rhoA": rhoA_substrate, "h": 1.0e-3},
    "piezo":     {"EI": EI_piezo,     "rhoA": rhoA_piezo,     "h": 1.0e-3},
}

region_sequence = ['piezo', 'substrate', 'piezo',  'substrate', 'piezo', 'substrate']

x_starts = np.array([0, 80, 85, 165, 170, 250]) * 1e-3
                    

geom = FE_module.build_geometry_from_types(L, region_types, region_sequence, x_starts)
# Automatically detects piezos at indices 1 and 4

base_params.geometry = geom

# # ============================================================
# # Build finite element model
# # ============================================================
# fe = FE_module.PiezoBeamFE(params_fe)

print("Region properties:")
for name, props in region_types.items():
    print(f"  {name:10s}: EI = {props['EI']:.4e} N m^2, rhoA = {props['rhoA']:.4e} kg/m")


## 3. Optimization configuration

The design vector is:

\[
\mathbf z=[g_0,\ell_1,g_1,\ell_2,\ldots,g_{N_p-1},\ell_{N_p}]
\]

where:

- `g0` is the substrate gap before patch 1,
- `ell_j` is the active piezo length of patch `j`,
- `g_j` is the active-piezo-to-active-piezo gap before the next patch,
- the final tip gap is whatever beam length remains.

Each active piezo region is surrounded by passive Kapton margins. The explicit regions are:

\[
\text{substrate}\rightarrow\text{kapton}\rightarrow\text{piezo}\rightarrow\text{kapton}\rightarrow\text{substrate}\rightarrow\cdots
\]


In [ ]:
@dataclass
class SimpleConstraints:
    # Fixed number of patches for this simple notebook.
    # To test different patch counts, change this and re-run.
    Np: int = 3

    # Active piezo length bounds [m]
    length_bounds: Tuple[float, float] = (5e-3, 40e-3)

    # Active-piezo gap bounds [m]
    root_gap_bounds: Tuple[float, float] = (0.0, 80e-3)
    internal_gap_bounds: Tuple[float, float] = (2e-3, 80e-3)
    tip_gap_bounds: Tuple[float, float] = (0.0, 80e-3)

    # Passive Kapton margin around each active piezo [m]
    kapton_left: float = 4.5e-3
    kapton_right: float = 4.5e-3

    # Optional total active piezo length bound [m]. Set to None to disable.
    total_active_length_bounds: Optional[Tuple[float, float]] = None

    # Optional fixed dimensions. Indices are zero-based.
    # Example: fixed_lengths={0: 28e-3} fixes patch 1 to 28 mm.
    fixed_lengths: Dict[int, float] = field(default_factory=dict)
    fixed_gaps_before: Dict[int, float] = field(default_factory=dict)  # gap before patch j

@dataclass
class SingleModeObjective:
    mode_index: int = 0                 # zero-based: 0 = first mode
    rel_freq_window: float = 0.25       # search +/- 25% around natural frequency
    min_half_window_hz: float = 0.25    # minimum half-width of search window
    n_freq: int = 80                    # frequency samples in inner search
    voltage_amplitude: float = 1.0      # response is per this voltage amplitude
    phase_type: str = "binary"          # "binary" or "continuous"
    max_binary_bruteforce_patches: int = 12

@dataclass
class OptimizerSettings:
    maxiter: int = 20
    popsize: int = 8
    seed: int = 1
    polish: bool = False
    workers: int = 1

constraints = SimpleConstraints(
    Np=3,
    length_bounds=(10e-3, 35e-3),
    root_gap_bounds=(0.0, 80e-3),
    internal_gap_bounds=(4e-3, 80e-3),
    tip_gap_bounds=(0.0, 80e-3),
    kapton_left=4.5e-3,
    kapton_right=4.5e-3,
    total_active_length_bounds=None,
    fixed_lengths={},
    fixed_gaps_before={},
)

mode_obj = SingleModeObjective(
    mode_index=0,
    rel_freq_window=0.30,
    min_half_window_hz=0.25,
    n_freq=80,
    voltage_amplitude=1.0,
    phase_type="binary",
)

opt_settings = OptimizerSettings(maxiter=12, popsize=6, seed=2, polish=False, workers=1)


## 4. Geometry decoding and region builder

This is the key replacement: instead of `build_geometry_arbitrary_piezos(...)`, this notebook builds explicit `regions` and calls:

```python
FE_module.build_geometry_with_regions(L=L, regions=regions, piezos=piezos)
```

Only the active `piezo` intervals are added to the `piezos` list. Kapton regions affect mass/stiffness but are not actuators.


In [ ]:
def make_bounds(c: SimpleConstraints) -> List[Tuple[float, float]]:
    """Bounds for z = [g0, ell1, g1, ell2, ..., g_{Np-1}, ell_Np]."""
    bounds = []
    for j in range(c.Np):
        # gap before patch j
        if j == 0:
            gb = c.root_gap_bounds
        else:
            gb = c.internal_gap_bounds
        if j in c.fixed_gaps_before:
            val = c.fixed_gaps_before[j]
            gb = (val, val)
        bounds.append(gb)

        # patch length j
        lb = c.length_bounds
        if j in c.fixed_lengths:
            val = c.fixed_lengths[j]
            lb = (val, val)
        bounds.append(lb)
    return bounds


def decode_design(z: np.ndarray, c: SimpleConstraints, L: float):
    """
    Convert optimizer vector z into active piezo edges xL, xR.
    Returns a dict with active piezo edges and tip gap.
    """
    z = np.asarray(z, dtype=float)
    assert len(z) == 2*c.Np

    xL = []
    xR = []
    cursor = 0.0

    for j in range(c.Np):
        gap_before = z[2*j]
        length = z[2*j + 1]
        x_left = cursor + gap_before
        x_right = x_left + length
        xL.append(x_left)
        xR.append(x_right)
        cursor = x_right

    xL = np.array(xL)
    xR = np.array(xR)
    tip_gap = L - xR[-1]

    return {"xL": xL, "xR": xR, "tip_gap": tip_gap}


def geometry_penalty(layout, c: SimpleConstraints, L: float, large: float = 1e12) -> float:
    """Return 0 if geometry is valid; otherwise return a large penalty."""
    xL = layout["xL"]
    xR = layout["xR"]
    tip_gap = layout["tip_gap"]

    if not np.all(np.isfinite(xL)) or not np.all(np.isfinite(xR)):
        return large
    if np.any(xL < 0.0) or np.any(xR > L):
        return large
    if np.any(xR <= xL):
        return large

    lengths = xR - xL
    if np.any(lengths < c.length_bounds[0]) or np.any(lengths > c.length_bounds[1]):
        return large

    # Active piezo gaps
    active_gaps = xL[1:] - xR[:-1]
    if len(active_gaps) > 0:
        if np.any(active_gaps < c.internal_gap_bounds[0]) or np.any(active_gaps > c.internal_gap_bounds[1]):
            return large

    if tip_gap < c.tip_gap_bounds[0] or tip_gap > c.tip_gap_bounds[1]:
        return large

    # Passive Kapton assembly overlap constraint.
    # If margins overlap, build_geometry_with_regions would create overlapping regions.
    assembly_left = np.maximum(0.0, xL - c.kapton_left)
    assembly_right = np.minimum(L, xR + c.kapton_right)
    if np.any(assembly_left[1:] < assembly_right[:-1]):
        return large

    if c.total_active_length_bounds is not None:
        total = np.sum(lengths)
        lo, hi = c.total_active_length_bounds
        if total < lo or total > hi:
            return large

    return 0.0


# def build_regions_with_kapton(layout, c: SimpleConstraints, L: float, region_types: Dict[str, dict]):
#     """
#     Build explicit substrate/kapton/piezo regions for one candidate layout.

#     Active piezo intervals [xL[j], xR[j]] are actuators.
#     Kapton regions affect EI and rhoA but are not actuators.
#     """
#     xL = layout["xL"]
#     xR = layout["xR"]

#     regions = []
#     piezos = []
#     cursor = 0.0

#     for j, (a, b) in enumerate(zip(xL, xR)):
#         kL = max(0.0, a - c.kapton_left)
#         kR = min(L, b + c.kapton_right)

#         # Substrate before this patch assembly
#         if kL > cursor + 1e-12:
#             regions.append({
#                 "x_start": cursor,
#                 "x_end": kL,
#                 "EI": region_types["substrate"]["EI"],
#                 "rhoA": region_types["substrate"]["rhoA"],
#                 "h": region_types["substrate"]["h"],
#                 "name": "substrate",
#             })

#         # Kapton left margin
#         if a > kL + 1e-12:
#             regions.append({
#                 "x_start": kL,
#                 "x_end": a,
#                 "EI": region_types["kapton"]["EI"],
#                 "rhoA": region_types["kapton"]["rhoA"],
#                 "h": region_types["kapton"]["h"],
#                 "name": "kapton_left",
#             })

#         # Active piezo region
#         regions.append({
#             "x_start": a,
#             "x_end": b,
#             "EI": region_types["piezo"]["EI"],
#             "rhoA": region_types["piezo"]["rhoA"],
#             "h": region_types["piezo"]["h"],
#             "name": "piezo",
#         })
#         piezos.append({"xL": float(a), "xR": float(b)})

#         # Kapton right margin
#         if kR > b + 1e-12:
#             regions.append({
#                 "x_start": b,
#                 "x_end": kR,
#                 "EI": region_types["kapton"]["EI"],
#                 "rhoA": region_types["kapton"]["rhoA"],
#                 "h": region_types["kapton"]["h"],
#                 "name": "kapton_right",
#             })

#         cursor = kR

#     # Substrate after final patch assembly
#     if cursor < L - 1e-12:
#         regions.append({
#             "x_start": cursor,
#             "x_end": L,
#             "EI": region_types["substrate"]["EI"],
#             "rhoA": region_types["substrate"]["rhoA"],
#             "h": region_types["substrate"]["h"],
#             "name": "substrate",
#         })

#     return regions, piezos


def build_fe_for_design(z: np.ndarray):
    """Decode z, build explicit regions, build FE model."""
    layout = decode_design(z, constraints, L)
    penalty = geometry_penalty(layout, constraints, L)
    if penalty > 0:
        return None, layout, penalty

    # regions, piezos = build_regions_with_kapton(layout, constraints, L, region_types)

    region_types = {
        "substrate": {"EI": EI_substrate, "rhoA": rhoA_substrate, "h": 1.0e-3},
        "piezo":     {"EI": EI_piezo,     "rhoA": rhoA_piezo,     "h": 1.0e-3},
    }

    region_sequence = ['piezo', 'substrate', 'piezo',  'substrate', 'piezo', 'substrate']

    x_starts = np.array([layout["xL"][0], layout["xR"][0], layout["xL"][1], layout["xR"][1], layout["xL"][2], layout["xR"][2]]) * 1e-3                   

    geom = FE_module.build_geometry_from_types(L, region_types, region_sequence, x_starts)
# Automatically detects piezos at indices 1 and 4
    # geom = FE_module.build_geometry_with_regions(
    #     L=L,
    #     regions=regions,
    #     piezos=piezos,
    #     default_h=1.0e-3,
    # )

    params = copy.deepcopy(base_params)
    params.geometry = geom
    fe = FE_module.PiezoBeamFE(params)
    return fe, layout, 0.0


## 5. Single-mode tip FRF evaluator

For a fixed geometry, the linear frequency-domain response is:

\[
\hat{u}(\omega)=\left(K+i\omega C-\omega^2M\right)^{-1}\Gamma \hat{v}.
\]

The tip response due to unit voltage on each patch is:

\[
\mathbf h(\omega)=e_{tip}^T\left(K+i\omega C-\omega^2M\right)^{-1}\Gamma.
\]

The inner layer chooses frequency and phase/sign pattern to maximize \(|\mathbf h\mathbf v|\).


In [ ]:
def damping_matrix(fe) -> np.ndarray:
    """
    Return mechanical damping matrix for the reduced mechanical system.
    Keep this simple: use modal damping reconstructed by FE3 if available.
    """
    if hasattr(fe, "C_red"):
        return fe.C_red
    return fe.params.c_alpha * fe.M_red + fe.params.c_beta * fe.K_red


def tip_reduced_index(fe) -> int:
    """Reduced DOF index corresponding to full tip transverse displacement."""
    full_tip_w_dof = 2 * (len(fe.geom.x_nodes) - 1)
    idx = np.where(fe.free_dofs == full_tip_w_dof)[0]
    if len(idx) != 1:
        raise RuntimeError("Could not find tip displacement DOF in reduced system.")
    return int(idx[0])


def patch_to_tip_contributions(fe, omega: float) -> np.ndarray:
    """
    Return h_j(omega): complex tip displacement from unit voltage on each patch.
    """
    K = fe.K_red
    M = fe.M_red
    C = damping_matrix(fe)
    Gamma = fe.Gamma_red

    Z = K + 1j * omega * C - (omega**2) * M
    U_cols = np.linalg.solve(Z, Gamma)   # shape: (Nfree, Np)

    return U_cols[tip_reduced_index(fe), :]


def best_phase_for_h(h: np.ndarray, obj: SingleModeObjective) -> dict:
    """
    Inner actuation optimization at one frequency.
    Supports binary +/- voltage or continuous phase alignment.
    """
    h = np.asarray(h, dtype=complex)
    Np = h.size
    A = obj.voltage_amplitude

    if obj.phase_type.lower() == "continuous":
        phases = -np.angle(h)
        v = A * np.exp(1j * phases)
        response = abs(np.dot(h, v))
        return {
            "response": float(response),
            "voltage_vector": v,
            "phases_rad": phases,
            "signs": None,
            "method": "continuous_phase_alignment",
        }

    if obj.phase_type.lower() == "binary":
        if Np > obj.max_binary_bruteforce_patches:
            raise ValueError("Too many patches for brute-force binary phase search in this simple skeleton.")

        best = None
        for signs in itertools.product([-1.0, 1.0], repeat=Np):
            signs = np.asarray(signs)
            v = A * signs.astype(complex)
            response = abs(np.dot(h, v))
            if best is None or response > best["response"]:
                best = {
                    "response": float(response),
                    "voltage_vector": v,
                    "phases_rad": np.where(signs > 0, 0.0, np.pi),
                    "signs": signs,
                    "method": "binary_bruteforce",
                }
        return best

    raise ValueError(f"Unknown phase_type={obj.phase_type!r}")


def frequency_grid_for_mode(fe, obj: SingleModeObjective) -> np.ndarray:
    """Build a frequency grid around the current geometry's target natural frequency."""
    f0 = float(fe.freq[obj.mode_index])
    half_width = max(obj.rel_freq_window * f0, obj.min_half_window_hz)
    f_min = max(1e-6, f0 - half_width)
    f_max = f0 + half_width
    return np.linspace(f_min, f_max, obj.n_freq)


def evaluate_single_mode_tip(fe, obj: SingleModeObjective) -> dict:
    """Inner layer: optimize frequency and phase for one mode at the tip."""
    freq_grid = frequency_grid_for_mode(fe, obj)
    best = None

    for f_hz in freq_grid:
        omega = 2.0 * np.pi * f_hz
        h = patch_to_tip_contributions(fe, omega)
        act = best_phase_for_h(h, obj)

        if best is None or act["response"] > best["response"]:
            best = {
                **act,
                "freq_hz": float(f_hz),
                "omega": float(omega),
                "h": h,
            }

    return best


## 6. Outer objective and optimizer

The optimizer minimizes a scalar. Since we want to maximize tip response, the objective returns the negative response.

Each objective call does:

\[
\mathbf z \rightarrow (x_L,x_R) \rightarrow \text{regions} \rightarrow \text{new FE model} \rightarrow \max_{\omega,\mathbf s}|y_{tip}|.
\]


In [ ]:
evaluation_history = []


def objective(z: np.ndarray) -> float:
    fe, layout, penalty = build_fe_for_design(z)
    if penalty > 0 or fe is None:
        return penalty

    try:
        inner = evaluate_single_mode_tip(fe, mode_obj)
        score = inner["response"]
    except Exception:
        # Failed geometries get rejected by a large positive objective.
        return 1e12

    evaluation_history.append({
        "z": np.asarray(z, dtype=float).copy(),
        "layout": layout,
        "score": score,
        "freq_hz": inner["freq_hz"],
        "signs": inner["signs"],
        "phases_rad": inner["phases_rad"],
    })

    return -score

bounds = make_bounds(constraints)
print("Number of optimization variables:", len(bounds))
print("Bounds:")
for i, bnd in enumerate(bounds):
    label = "gap" if i % 2 == 0 else "length"
    print(f"  z[{i}] ({label:6s}) = {bnd}")

result = differential_evolution(
    objective,
    bounds=bounds,
    maxiter=opt_settings.maxiter,
    popsize=opt_settings.popsize,
    seed=opt_settings.seed,
    polish=opt_settings.polish,
    workers=opt_settings.workers,
    updating="immediate" if opt_settings.workers == 1 else "deferred",
    disp=True,
)

print("\nOptimization result:")
print(result)


## 7. Inspect the best result


In [ ]:
best_z = result.x
best_fe, best_layout, best_penalty = build_fe_for_design(best_z)
best_inner = evaluate_single_mode_tip(best_fe, mode_obj)

print("Best objective response [m/V or m for given voltage]:", best_inner["response"])
print("Best actuation frequency [Hz]:", best_inner["freq_hz"])
print("Natural frequencies of best geometry [Hz]:", best_fe.freq[:5])
print("Active xL [mm]:", 1e3 * best_layout["xL"])
print("Active xR [mm]:", 1e3 * best_layout["xR"])
print("Active lengths [mm]:", 1e3 * (best_layout["xR"] - best_layout["xL"]))
print("Tip gap [mm]:", 1e3 * best_layout["tip_gap"])
print("Binary signs:", best_inner["signs"])
print("Phases [deg]:", np.rad2deg(best_inner["phases_rad"]))


## 8. Plot the best layout and FRF around the target mode


In [ ]:
def plot_layout(layout, L):
    fig, ax = plt.subplots(figsize=(10, 1.8))
    ax.plot([0, L], [0, 0], "k-", lw=3, label="beam")
    for j, (a, b) in enumerate(zip(layout["xL"], layout["xR"])):
        ax.axvspan(a, b, alpha=0.5, label="active piezo" if j == 0 else None)
        ax.text(0.5*(a+b), 0.02, f"P{j+1}", ha="center", va="bottom")
    ax.set_xlim(0, L)
    ax.set_yticks([])
    ax.set_xlabel("x [m]")
    ax.legend(loc="upper right")
    ax.set_title("Optimized active piezo layout")
    plt.show()

plot_layout(best_layout, L)

freq_grid = frequency_grid_for_mode(best_fe, mode_obj)
resp = []
for f_hz in freq_grid:
    h = patch_to_tip_contributions(best_fe, 2*np.pi*f_hz)
    resp.append(best_phase_for_h(h, mode_obj)["response"])
resp = np.array(resp)

plt.figure(figsize=(8, 4))
plt.plot(freq_grid, resp, lw=2)
plt.axvline(best_inner["freq_hz"], ls="--", label=f"best {best_inner['freq_hz']:.3f} Hz")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Best tip response")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 9. Notes for extending this skeleton

- To optimize another mode, change `mode_obj.mode_index`.
- To test multiple patch counts, rerun the notebook with different `constraints.Np` values.
- To make this multi-mode again, evaluate `evaluate_single_mode_tip(...)` for several `mode_index` values and sum/normalize the responses.
- To use average/RMS beam deflection instead of tip deflection, replace `patch_to_tip_contributions(...)` with a function that returns all transverse displacement rows and computes mean/RMS after applying the voltage vector.
- To include fixed patch sizes/gaps, use `fixed_lengths` and `fixed_gaps_before` in `SimpleConstraints`.
